In [1]:
# Patch gevent for proper async handling
from gevent import monkey
monkey.patch_all()

# Import mu_bench modules
from mu_bench.metrics_exporter.prometheus_exporter import PrometheusExporter
from mu_bench.request_generator.edge_request_generator import EdgeRequestGenerator
from mu_bench.experiment_manager import EdgeExperimentManager

In [2]:
config = {
    "base_host": "172.21.0.5",
    "apps": [
        {
            "name": "app-iot-gateway",  # Different name creates different hash
            "port": 31115,
            "traffic_weight": 1,
            "wait_time": {"min": 0.7, "max": 1.5}
        },
        # {
        #     "name": "app-edge-2",  # Different name = different behavior
        #     "port": 31116,
        #     "traffic_weight": 0.9,
        #     "wait_time": {"min": 0.8, "max": 2.5}
        # },
    ],
    "edge_sites": {
        "edge-site-1": {
            "pattern": "urban_peak",
            "description": "Urban site with business peaks",
            "traffic_weight": 1.0
        },
        "edge-site-2": {
            "pattern": "suburban_steady", 
            "description": "Suburban site with steady traffic",
            "traffic_weight": 0.8
        }
    },
    "endpoint": "iot-device-gateway"
}

In [3]:
prometheus_logger = PrometheusExporter(
    prometheus_url="http://localhost:9090",
    target_namespaces=["app-iot-gateway"],
    collection_interval_minutes=5,
    metrics_step="30s",
    output_dir="metrics",
    verbose=True,
    create_output_dir=False
)

# Create traffic generator
traffic_generator = EdgeRequestGenerator()

# Create experiment manager
experiment_manager = EdgeExperimentManager(
    prometheus_exporter=prometheus_logger,
    traffic_generator=traffic_generator,
    experiment_name=None,
    output_base_dir="../DATA"
)

# Run experiment

try:
    success = experiment_manager.run_experiment(
        edge_config=config,
        duration_hours=0.01,        # Real time
        simulated_hours=24,      # Simulated time: 24 hours
        max_users=2,
        spawn_rate=1
    )
except Exception as e:
    print("\n🚫 Experiment interrupted by user")
finally:
    if success:
        print(f"\n🎉 Experiment completed! Results: {experiment_manager.output_directory}")
    else:
        print("\n💥 Experiment failed!")


🚀 EXPERIMENT: edge_experiment_2025_10_16_14_36
📋 CONFIGURATION
Duration:       0.01h real → 24h simulated
Host:           172.21.0.5
Output:         edge_experiment_2025_10_16_14_36
Metrics:        Every 5min

🔌 APPLICATIONS (1)
  app-iot-gateway Port 31115  Weight 1

🌍 EDGE SITES (2)
  edge-site-1     urban_peak      Weight 1.0
  edge-site-2     suburban_steady Weight 0.8

📈 ENDPOINTS: 2 total
Starting metrics collection every 5 minutes
Metrics step interval: 30s
Output directory: ../DATA/edge_experiment_2025_10_16_14_36/metrics
Using 30s step intervals for the past 5 minutes
Metrics collection started
Metrics collection completed at 2025-10-16T11:36:29.916613+00:00


Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated                                                                         0     0(0.00%) |      0       0       0      0 |    0.00        0.00



GlobalUserCountShape: Simulated time reset to 0.00s.


Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
GET      31115:/iot-device-gateway-edge-site-1                                              2     0(0.00%) |    114     107     121    110 |    0.00        0.00
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated                                                                         2     0(0.00%) |    114     107     121    110 |    0.00        0.00

Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|------------------------

Stopping metrics collection...
Stopping metrics collection...
Metrics collection stopped
📊 Metrics collection completed

💥 Experiment failed!
